In [ ]:
BSPT_project/
├── data/
│   ├── raw/
│   │   └── landsat_data.csv               # Raw Landsat 5, 7 & 8 data
│   └── processed/
│       └── scaled_data.csv                # Preprocessed and scaled data
├── models/
│   ├── vision_transformer.py              # Vision Transformer module
│   ├── transformer_xl.py                  # Transformer-XL module
│   ├── fusion_module.py                   # Fusion module
│   ├── beam_search.py                     # Beam Search module
│   └── mlp.py                             # MLP module
├── utils/
│   ├── preprocessing.py                   # Data preprocessing functions
│   ├── evaluation.py                      # Evaluation metrics
│   ├── visualization.py                   # Plotting functions
│   └── shap_analysis.py                   # SHAP analysis functions
├── main.py                                # Main script to run training & evaluation
├── requirements.txt                       # All pip packages
└── README.md                              # Project overview and usage


In [ ]:
pandas
numpy
matplotlib
seaborn
shap
scikit-learn
torch
torchvision
transformers
networkx


In [ ]:
pip install -r requirements.txt


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import shap
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import KFold
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from torchvision.models.vision_transformer import VisionTransformer
from transformers import TransfoXLModel, TransfoXLConfig

In [ ]:
# ====================== Parameters =======================
BATCH_SIZE = 32
EPOCHS = 15
LEARNING_RATE = 0.01
VALIDATION_SPLIT = 0.2
EARLY_STOPPING = True
BEAM_WIDTH = 3
NODE_SCORE = 800
TREE_LEVELS = 5

In [ ]:
# ====================== Data Preprocessing =======================
def preprocess_data(file_path):
    df = pd.read_csv(file_path)
    numeric_cols = df.select_dtypes(include=['number']).columns
    imputer = SimpleImputer(strategy='mean')
    scaler = MinMaxScaler()
    df[numeric_cols] = imputer.fit_transform(df[numeric_cols])
    df[numeric_cols] = scaler.fit_transform(df[numeric_cols])
    return df

# ====================== Model Definitions =======================
class ViTModule(nn.Module):
    def __init__(self):
        super(ViTModule, self).__init__()
        self.vit = VisionTransformer(
            image_size=224,
            patch_size=16,
            num_layers=12,
            num_heads=12,
            hidden_dim=768,
            mlp_dim=3072,
            dropout=0.1
        )

    def forward(self, x):
        return self.vit(x)

class TransformerXLModule(nn.Module):
    def __init__(self):
        super(TransformerXLModule, self).__init__()
        config = TransfoXLConfig(
            d_model=512,
            mem_len=512,
            n_layer=12,
            n_head=8,
            d_inner=2048
        )
        self.transformer_xl = TransfoXLModel(config)

    def forward(self, x, mems=None):
        output = self.transformer_xl(input_ids=x, mems=mems)
        return output.last_hidden_state

class FusionModule(nn.Module):
    def __init__(self):
        super(FusionModule, self).__init__()
        self.attention = nn.MultiheadAttention(embed_dim=1024, num_heads=16)
        self.linear = nn.Linear(1024, 1024)

    def forward(self, spatial_features, temporal_features):
        combined = torch.cat((spatial_features, temporal_features), dim=1)
        attn_output, _ = self.attention(combined, combined, combined)
        return self.linear(attn_output)

class MLPModule(nn.Module):
    def __init__(self, input_dim=1024):
        super(MLPModule, self).__init__()
        self.fc1 = nn.Linear(input_dim, 32)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.1)
        self.fc2 = nn.Linear(32, 1)

    def forward(self, x):
        x = self.relu(self.fc1(x))
        x = self.dropout(x)
        return self.fc2(x)

# ====================== Beam Search =======================
def beam_search(model, input_tensor, beam_width=BEAM_WIDTH, levels=TREE_LEVELS):
    sequences = [[list(), 0.0]]
    for _ in range(levels):
        all_candidates = []
        for seq, score in sequences:
            for i in range(beam_width):
                candidate = seq + [i]
                input_seq = torch.tensor(candidate).unsqueeze(0).float()
                output = model(input_tensor)
                candidate_score = -output.mean().item()
                all_candidates.append((candidate, score + candidate_score))
        ordered = sorted(all_candidates, key=lambda tup: tup[1])
        sequences = ordered[:beam_width]
    return sequences[0][0]

# ====================== Evaluation & Visualization =======================
def evaluate(y_true, y_pred):
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    return {'MSE': mse, 'RMSE': rmse, 'R2': r2, 'MAE': mae}

def plot_scatter(y_true, y_pred):
    sns.regplot(x=y_true, y=y_pred, line_kws={"color": "red"})
    plt.xlabel("Actual")
    plt.ylabel("Predicted")
    plt.title("Actual vs Predicted")
    m, b = np.polyfit(y_true, y_pred, 1)
    plt.text(min(y_true), max(y_pred), f'y={m:.2f}x+{b:.2f}', color='red')
    plt.show()

def plot_municipal_performance(metrics_dict):
    df = pd.DataFrame(metrics_dict).T
    df.plot(kind='bar')
    plt.title("Municipal Performance")
    plt.ylabel("Score")
    plt.show()

def shap_summary_plot(model, X):
    explainer = shap.Explainer(model, X)
    shap_values = explainer(X)
    shap.summary_plot(shap_values, X)

def plot_cumulative_gain(y_true, y_pred):
    sorted_indices = np.argsort(y_pred)[::-1]
    y_sorted = np.array(y_true)[sorted_indices]
    cumulative_gain = np.cumsum(y_sorted) / np.sum(y_sorted)
    plt.plot(np.linspace(0, 1, len(cumulative_gain)), cumulative_gain)
    plt.title("Cumulative Gain Chart")
    plt.xlabel("Fraction of Samples")
    plt.ylabel("Cumulative Gain")
    plt.grid(True)
    plt.show()

# ====================== Main Execution =======================
def main():
    data = preprocess_data('data/raw/landsat_data.csv')
    X = data.drop('yield', axis=1).values
    y = data['yield'].values

    X_tensor = torch.tensor(X, dtype=torch.float32)
    y_tensor = torch.tensor(y, dtype=torch.float32).unsqueeze(1)

    kf = KFold(n_splits=5)
    metrics = {}

    for fold, (train_idx, test_idx) in enumerate(kf.split(X_tensor)):
        X_train, X_test = X_tensor[train_idx], X_tensor[test_idx]
        y_train, y_test = y_tensor[train_idx], y_tensor[test_idx]

        model = MLPModule(input_dim=X_train.shape[1])
        criterion = nn.MSELoss()
        optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

        train_dataset = TensorDataset(X_train, y_train)
        train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)

        for epoch in range(EPOCHS):
            model.train()
            for batch_X, batch_y in train_loader:
                optimizer.zero_grad()
                outputs = model(batch_X)
                loss = criterion(outputs, batch_y)
                loss.backward()
                optimizer.step()

        model.eval()
        with torch.no_grad():
            y_pred = model(X_test).detach().numpy()

        fold_metrics = evaluate(y_test.detach().numpy(), y_pred)
        metrics[f'Fold {fold+1}'] = fold_metrics

        print(f"Fold {fold+1} Metrics:", fold_metrics)
        plot_scatter(y_test.detach().numpy(), y_pred)
        plot_cumulative_gain(y_test.detach().numpy(), y_pred)

    plot_municipal_performance(metrics)
    shap_summary_plot(model, X_tensor)

    # Beam Search Example (placeholder usage)
    best_sequence = beam_search(model, X_tensor[:1])
    print("Best node sequence from Beam Search:", best_sequence)

if __name__ == "__main__":
    main()